# Parameter Shifting

Sweeps a single parameter across a range of values and plots behavioral benchmarks
(SPC, CRP, PNR) for each value. Dispatched via papermill from `render_parameter_shifting.ipynb`.

In [ ]:
# Papermill parameters
varied_parameter = "start_drift_rate"
sweep_min = 0.0
sweep_max = 1.0
model_name = "BaseCMR"
model_factory_path = "cru_to_cmr.models.cmr_compare.CMRFactory"
fit_result_name = "HealeyKahana2014_BaseCMR_Fitting.json"
data_name = "HealeyKahana2014"
data_query = "data['listtype'] == -1"
experiment_count = 50
run_tag = "Parameter_Shifting"
seed = 0
analysis_paths = [
    "jaxcmr.analyses.spc.plot_spc",
    "jaxcmr.analyses.crp.plot_crp",
    "jaxcmr.analyses.pnr.plot_pnr",
]

In [ ]:
# Parameters
model_name = "BaseCMR"
model_factory_path = "cru_to_cmr.models.cmr_compare.CMRFactory"
fit_result_name = "HealeyKahana2014_BaseCMR_Fitting.json"
data_name = "HealeyKahana2014"
data_query = "data['listtype'] == -1"
experiment_count = 50
run_tag = "Parameter_Shifting"
seed = 0
analysis_paths = ["jaxcmr.analyses.spc.plot_spc", "jaxcmr.analyses.crp.plot_crp", "jaxcmr.analyses.pnr.plot_pnr"]
varied_parameter = "primacy_scale"
sweep_min = 1.1920928955078125e-07
sweep_max = 99.99999988079071


In [ ]:
import json
import os

import jax.numpy as jnp
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from jax import random
from matplotlib import rcParams

from jaxcmr.helpers import (
    find_project_root,
    format_floats,
    generate_trial_mask,
    import_from_string,
    load_data,
)
from jaxcmr.simulation import parameter_shifted_simulate_h5_from_h5

ROOT = find_project_root()

# Load data
data = load_data(f"{ROOT}/data/{data_name}.h5")
trial_mask = generate_trial_mask(data, data_query)

# Load model factory
model_factory = import_from_string(model_factory_path)

# Load analysis functions
analyses = [import_from_string(path) for path in analysis_paths]

# Connections matrix (zeros — no semantic similarity for this project)
max_size = np.max(data["pres_itemnos"])
connections = jnp.zeros((max_size, max_size))

# Load pre-fit parameters
fit_path = f"{ROOT}/fits/{fit_result_name}"
with open(fit_path) as f:
    fit_result = json.load(f)
    if "subject" not in fit_result["fits"]:
        fit_result["fits"]["subject"] = fit_result["subject"]

# Output directory
target_directory = f"{ROOT}/figures/shifting/"
os.makedirs(target_directory, exist_ok=True)

rng = random.PRNGKey(seed)

In [ ]:
# Compute sweep values using matplotlib color cycle length
color_cycle = [each["color"] for each in rcParams["axes.prop_cycle"]]
considered_values = jnp.linspace(
    start=sweep_min, stop=sweep_max, num=len(color_cycle)
)[:-1].tolist()

# Simulate
rng, rng_iter = random.split(rng)
params = {key: jnp.array(val) for key, val in fit_result["fits"].items()}

sim = parameter_shifted_simulate_h5_from_h5(
    model_factory=model_factory,
    dataset=data,
    features=connections,
    parameters=params,
    trial_mask=trial_mask,
    experiment_count=experiment_count,
    varied_parameter=varied_parameter,
    parameter_values=considered_values,
    rng=rng_iter,
)

# Reset subjects for uniformity
for i in range(len(sim)):
    sim[i]["subject"] *= 0

sim_trial_mask = generate_trial_mask(sim[0], data_query)

In [ ]:
# Generate greyscale figures for each analysis
for analysis in analyses:
    figure_str = (
        f"bw_{model_name}_{varied_parameter.title()}_{run_tag}_"
        f"{analysis.__name__[5:]}_{data_name}.png"
    )
    figure_path = f"{target_directory}{figure_str}"
    print(f"Saving: {figure_path}")

    # Greyscale color cycle
    cmap = plt.get_cmap("Greys")
    n_vals = len(considered_values)
    eps = 0.15
    grey_cycle = [mcolors.rgb2hex(cmap(x)) for x in np.linspace(eps, 1 - eps, n_vals)]

    axis = analysis(
        datasets=sim,
        trial_masks=[sim_trial_mask] * len(considered_values),
        color_cycle=grey_cycle,
        distances=1 - connections,
        axis=plt.gca(),
        labels=format_floats(considered_values, 1),
        contrast_name=varied_parameter,
    )
    axis.get_lines()[-1].set_linewidth(2.5)

    axis.tick_params(labelsize=14)
    axis.set_xlabel(axis.get_xlabel(), fontsize=16)
    axis.set_ylabel(axis.get_ylabel(), fontsize=16)
    axis.legend(loc="upper left", bbox_to_anchor=(1, 1))

    plt.savefig(figure_path, bbox_inches="tight", dpi=600)
    plt.show()